# Winner + Pseudo-labeling

Pipeline em duas fases sobre BERTimbau para classificacao de laudos de mamografia.

- Fase 1: treino 5-fold como o winner (5 epochs, LR 2e-5, focal loss, FP16).
- Fase 2: predizer test, selecionar predicoes com confianca maior que 0.95.
- Retreinar (3 epochs, LR menor) com train + pseudo-labels, partindo do melhor estado da fase 1.
- Inferencia final com os novos modelos.
- Meta: +0.5 a +1 pp vs winner na macro F1.

Calibracao fixa: temperatura 0.958 e thresholds por classe herdados do winner.

In [ ]:
import os
import re
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

MAX_LEN      = 512
BATCH_SIZE   = 8
EPOCHS       = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS      = 5
SEED         = 42
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25
NUM_CLASSES  = 7

BEST_TEMP       = 0.958
BEST_THRESHOLDS = [0.95, 1.6, 1.35, 1.0, 0.4, 1.15, 0.1]

PSEUDO_CONFIDENCE = 0.95
PSEUDO_EPOCHS     = 3

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ==========================================================================
# AUTO-DETECTAR PATH DO MODELO NO KAGGLE
# ==========================================================================
import os
from pathlib import Path

def find_model_path():
    candidates = [
        '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
        '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
        '/kaggle/input/bert-large-portuguese-cased',
        '/kaggle/input/bertimbau-ptbr-complete',
        '/kaggle/input/bertimbau-large',
        '/kaggle/input/bertimbau',
        '/kaggle/input/neuralmind-bert-large-portuguese-cased',
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'config.json')):
            return c
    # Fallback: varrer /kaggle/input procurando config.json com model BERT grande
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    best = None
    for cfg_path in root.rglob('config.json'):
        try:
            with open(cfg_path) as f:
                cfg = json.load(f)
            # BERT large tipicamente tem hidden_size=1024
            if cfg.get('model_type') == 'bert' and cfg.get('hidden_size', 0) >= 1024:
                # Verifica se tem pesos e tokenizer no mesmo diretorio
                parent = cfg_path.parent
                has_tokenizer = any((parent / f).exists() for f in ['tokenizer.json', 'vocab.txt', 'tokenizer_config.json'])
                has_weights = any(p.suffix in ('.bin', '.safetensors') for p in parent.iterdir())
                if has_tokenizer and has_weights:
                    best = str(parent)
                    break
        except Exception:
            continue
    return best

import json as _json
MODEL_PATH = find_model_path()
assert MODEL_PATH is not None, (
    'Modelo BERTimbau Large nao encontrado em /kaggle/input.\n'
    'Adicione como Input: neuralmind/bert-large-portuguese-cased ou equivalente.\n'
    'Arquivos em /kaggle/input:\n  ' + '\n  '.join(
        str(p) for p in Path('/kaggle/input').rglob('config.json')
    )
)
print('MODEL_PATH detectado:', MODEL_PATH)
print('Arquivos:', sorted(os.listdir(MODEL_PATH)))


In [ ]:
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print('Train:', train_df.shape, 'Test:', test_df.shape)

train_texts  = [build_input_text(t) for t in train_df['report'].fillna('').tolist()]
train_labels = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].fillna('').tolist()]
print('Exemplo train_text[0]:')
print(train_texts[0][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class MammoDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            return_token_type_ids=True,
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc['token_type_ids'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none')
        pt = torch.exp(-ce)
        loss = self.alpha * (1 - pt) ** self.gamma * ce
        return loss.mean()

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, loss_fn):
    model.train()
    total = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        tti = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)
        with autocast():
            out = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tti)
            loss = loss_fn(out.logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / max(1, len(loader))

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        tti = batch['token_type_ids'].to(device)
        with autocast():
            out = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tti)
        p = F.softmax(out.logits.float(), dim=-1).cpu().numpy()
        probs.append(p)
    return np.concatenate(probs, axis=0)

def temperature_scale(probs, T):
    eps = 1e-12
    logp = np.log(np.clip(probs, eps, 1.0)) / T
    logp = logp - logp.max(axis=1, keepdims=True)
    e = np.exp(logp)
    return e / e.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adj = probs * np.array(thresholds, dtype=np.float32)
    return adj.argmax(axis=1)

def compute_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(train_texts, train_labels))

oof_probs = np.zeros((len(train_texts), NUM_CLASSES), dtype=np.float32)
test_probs = np.zeros((len(test_texts), NUM_CLASSES), dtype=np.float32)
fold_states = []

loss_fn = FocalLoss()

test_ds = MammoDataset(test_texts, None, tokenizer)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'\n=== Fase 1, Fold {fold+1}/{N_FOLDS} ===')
    set_seed(SEED + fold)
    tr_texts  = [train_texts[i] for i in tr_idx]
    tr_labels = [train_labels[i] for i in tr_idx]
    va_texts  = [train_texts[i] for i in va_idx]
    va_labels = [train_labels[i] for i in va_idx]

    tr_ds = MammoDataset(tr_texts, tr_labels, tokenizer)
    va_ds = MammoDataset(va_texts, va_labels, tokenizer)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=NUM_CLASSES, local_files_only=True
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(tr_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps
    )
    scaler = GradScaler()

    best_f1 = -1.0
    best_state = None
    for epoch in range(EPOCHS):
        tl = train_epoch(model, tr_loader, optimizer, scheduler, scaler, loss_fn)
        vp = predict(model, va_loader)
        vp_cal = temperature_scale(vp, BEST_TEMP)
        vpred = apply_thresholds(vp_cal, BEST_THRESHOLDS)
        f1 = compute_f1(va_labels, vpred)
        print(f'  epoch {epoch+1} loss={tl:.4f} val_f1={f1:.5f}')
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    oof_probs[va_idx] = predict(model, va_loader)
    test_probs += predict(model, test_loader) / N_FOLDS
    fold_states.append(best_state)

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

oof_cal = temperature_scale(oof_probs, BEST_TEMP)
oof_preds = apply_thresholds(oof_cal, BEST_THRESHOLDS)
oof_f1_phase1 = compute_f1(train_labels, oof_preds)
print(f'\nPhase 1 OOF F1: {oof_f1_phase1:.5f}')

In [ ]:
test_cal = temperature_scale(test_probs, BEST_TEMP)
test_preds_cal = apply_thresholds(test_cal, BEST_THRESHOLDS)
test_confidence = test_cal.max(axis=1)

pseudo_mask = test_confidence > PSEUDO_CONFIDENCE
n_pseudo = int(pseudo_mask.sum())
print(f'Pseudo-labels selected: {n_pseudo}/{len(test_texts)} ({n_pseudo/len(test_texts)*100:.1f}%)')

pseudo_labels = test_preds_cal[pseudo_mask]
pseudo_texts  = [test_texts[i] for i in range(len(test_texts)) if pseudo_mask[i]]
print('Pseudo class dist:', pd.Series(pseudo_labels).value_counts().sort_index().to_dict())

In [ ]:
phase2_test_probs = np.zeros((len(test_texts), NUM_CLASSES), dtype=np.float32)
phase2_oof_probs  = np.zeros((len(train_texts), NUM_CLASSES), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'\n=== Fase 2, Fold {fold+1}/{N_FOLDS} ===')
    set_seed(SEED + 100 + fold)

    combined_texts  = [train_texts[i] for i in tr_idx] + pseudo_texts
    combined_labels = [train_labels[i] for i in tr_idx] + pseudo_labels.tolist()

    va_texts  = [train_texts[i] for i in va_idx]
    va_labels = [train_labels[i] for i in va_idx]

    tr_ds = MammoDataset(combined_texts, combined_labels, tokenizer)
    va_ds = MammoDataset(va_texts, va_labels, tokenizer)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=NUM_CLASSES, local_files_only=True
    ).to(device)
    model.load_state_dict(fold_states[fold])

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR / 2, weight_decay=WEIGHT_DECAY)
    total_steps = len(tr_loader) * PSEUDO_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps
    )
    scaler = GradScaler()

    best_f1 = -1.0
    best_state = None
    for epoch in range(PSEUDO_EPOCHS):
        tl = train_epoch(model, tr_loader, optimizer, scheduler, scaler, loss_fn)
        vp = predict(model, va_loader)
        vp_cal = temperature_scale(vp, BEST_TEMP)
        vpred = apply_thresholds(vp_cal, BEST_THRESHOLDS)
        f1 = compute_f1(va_labels, vpred)
        print(f'  epoch {epoch+1} loss={tl:.4f} val_f1={f1:.5f}')
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    phase2_oof_probs[va_idx] = predict(model, va_loader)
    phase2_test_probs += predict(model, test_loader) / N_FOLDS

    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
oof_cal2 = temperature_scale(phase2_oof_probs, BEST_TEMP)
oof_preds2 = apply_thresholds(oof_cal2, BEST_THRESHOLDS)
oof_f1_phase2 = f1_score(train_labels, oof_preds2, average='macro')
print(f'Phase 1 OOF F1: {oof_f1_phase1:.5f}')
print(f'Phase 2 OOF F1: {oof_f1_phase2:.5f}')
print(f'Delta: {(oof_f1_phase2 - oof_f1_phase1)*100:+.2f} pp')

In [ ]:
test_cal2 = temperature_scale(phase2_test_probs, BEST_TEMP)
test_preds_final = apply_thresholds(test_cal2, BEST_THRESHOLDS)
submission = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds_final})
submission.to_csv('submission.csv', index=False)
print('Submission shape:', submission.shape)
print(submission.head())
print('Pred class dist:', submission['target'].value_counts().sort_index().to_dict())